# Multi-agent GDS generation — fine-tuned coder + Deep Agents v0.6

**Why a hybrid?** Qwen3.5's architecture uses linear/gated attention that llama.cpp (Ollama) can't run — so the fine-tuned 4B GGUF fails in Ollama (`qwen3next: layer 32 missing projections`). But it runs fine via Python/Unsloth.

So we split roles:

| Role | Model | Runs via |
|---|---|---|
| **Coder** (your specialist) | fine-tuned Qwen3.5-4B | Python / Unsloth (in-process) |
| Orchestrator + planner + validator | `qwen3.5:9b` | Ollama (needs tool-calling) |

**The fine-tuned model is exposed as a TOOL** `generate_glayout_code(plan)`. The orchestrator calls it for the part that matters — the actual code generation it was trained on — and uses the base model only to route between steps.

**Budget:** the coder tool is called fresh each time → its own 4096-token budget per call. The orchestrator can call it again with "continue from line N" if code is truncated, so total output is unbounded.

```
  user request
      │
      ▼
  orchestrator (qwen3.5:9b, Ollama)
      │
      ├─ task('planner')              → short plan
      ├─ tool generate_glayout_code() → FINE-TUNED 4B writes code   ← your model
      ├─ tool validate_python()       → ast check
      ├─ tool generate_glayout_code() → continue if truncated       ← your model again
      └─ tool execute_glayout()       → run → .gds → .png
```

## 1 · Config + verify Ollama

In [7]:
import os, sys, json, subprocess, ast, traceback, re, gc
from pathlib import Path

WORKSPACE = Path.cwd().parent.parent
sys.path.insert(0, str(WORKSPACE / "src" / "gelochip"))
os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/pdks"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")

ORCHESTRATOR_MODEL = "qwen3.5:9b"     # Ollama, tool-capable, runs the agent loop
FINETUNED_ADAPTER  = Path.cwd() / "qwen35_4b_gds_lora" / "lora_adapter"  # your specialist
OUT_DIR            = Path.cwd() / "agent_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CODER_MAX_TOK = 4096   # fresh budget every time the coder tool is called

r = subprocess.run(["ollama", "list"], capture_output=True, text=True)
available = [l.split()[0] for l in r.stdout.splitlines()[1:] if l.strip()]
print("Ollama models   :", available)
print(f"Orchestrator    : {ORCHESTRATOR_MODEL}  ({'✓' if ORCHESTRATOR_MODEL in available else '✗ run: ollama pull '+ORCHESTRATOR_MODEL})")
print(f"Fine-tuned coder: {FINETUNED_ADAPTER}  ({'✓' if FINETUNED_ADAPTER.exists() else '✗ not found'})")

Ollama models   : ['gds-qwen35-4b:latest', 'glm-5:cloud', 'qwen3.5:9b']
Orchestrator    : qwen3.5:9b  (✓)
Fine-tuned coder: /home/irman/Gelochip/notebooks/sft_finetuning/qwen35_4b_gds_lora/lora_adapter  (✓)


## 2 · Build the chat model

In [8]:
import torch
from langchain_ollama import ChatOllama

# ── Orchestrator: base qwen3.5:9b via Ollama (tool-calling) ──────────────────
llm = ChatOllama(model=ORCHESTRATOR_MODEL, temperature=0.2, num_ctx=8192, num_predict=2048)
print("Orchestrator smoke test:", llm.invoke("Reply with one word: ok").content[:40])

# ── Coder: your fine-tuned Qwen3.5-4B via Unsloth (in-process) ───────────────
from unsloth import FastVisionModel
import unsloth.models._utils as _u
_u._get_statistics = lambda *a, **k: None
_u.get_statistics  = lambda *a, **k: None
import bitsandbytes as bnb
if not getattr(bnb.nn.Params4bit, "_patched_unsloth", False):
    _orig_new = bnb.nn.Params4bit.__new__
    bnb.nn.Params4bit.__new__ = lambda cls, *a, **k: (k.pop("_is_hf_initialized", None), _orig_new(cls, *a, **k))[1]
    bnb.nn.Params4bit._patched_unsloth = True

gc.collect(); torch.cuda.empty_cache()
coder_model, coder_tok = FastVisionModel.from_pretrained(
    str(FINETUNED_ADAPTER), load_in_4bit=True,
    use_gradient_checkpointing=False, max_seq_length=4096,
)
FastVisionModel.for_inference(coder_model)
print(f"Fine-tuned coder loaded. GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Orchestrator smoke test: ok


Current model requires 320 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


==((====))==  Unsloth 2026.5.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.616 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Fine-tuned coder loaded. GPU mem: 0.01 GB


## 3 · Tools (the validator and executor)

These are real Python functions agents can call. Each is intentionally narrow.

In [9]:
from langchain_core.tools import tool

# ── THE fine-tuned model, exposed as a tool ──────────────────────────────────
def _coder_generate(instruction: str) -> str:
    """Run the fine-tuned 4B. Prefilled with ```python so it goes straight to code."""
    msgs = [{"role": "user", "content": [{"type": "text", "text": instruction}]}]
    chat = coder_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    chat += "<think>\n\n</think>\n\n```python\n"   # force code mode
    text_tok = getattr(coder_tok, "tokenizer", coder_tok)
    # Send inputs to wherever the model's input embedding actually lives — NOT a
    # hardcoded "cuda". With only ~7.6 GB VRAM (and Ollama's qwen3.5:9b already
    # resident), the 4-bit coder gets offloaded to CPU, so forcing inputs onto
    # cuda crashes with "index is on cuda:0, different from other tensors on cpu".
    embed_device = coder_model.get_input_embeddings().weight.device
    inputs = coder_tok(text=[chat], return_tensors="pt", padding=True).to(embed_device)
    with torch.inference_mode():
        out = coder_model.generate(
            **inputs, max_new_tokens=CODER_MAX_TOK, temperature=0.2, top_p=0.95,
            do_sample=True,
            pad_token_id=getattr(text_tok, "eos_token_id", 0),
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    body = text_tok.decode(gen, skip_special_tokens=True)
    code = "```python\n" + body
    m = re.search(r"```(?:python)?\s*\n(.*?)```", code, re.DOTALL) or \
        re.search(r"```(?:python)?\s*\n(.*)", code, re.DOTALL)
    return m.group(1).strip() if m else body.strip()

@tool
def generate_glayout_code(plan: str) -> str:
    """Generate glayout Python code for a circuit, using the fine-tuned specialist model.
    `plan` is a description / plan of the circuit. Returns Python code (may be truncated at 4096 tok)."""
    return _coder_generate(
        f"Generate complete glayout Python code on gf180 PDK for this circuit.\n\nPlan:\n{plan}"
    )

@tool
def continue_glayout_code(partial_code: str) -> str:
    """Continue generating glayout code that was cut off. Pass the last ~30 lines of existing code;
    returns the continuation only (no overlap)."""
    return _coder_generate(
        "Continue this glayout code from exactly where it stops. Output ONLY the continuation, "
        f"no repetition:\n\n{partial_code}"
    )

# ── Plain Python tools ───────────────────────────────────────────────────────
@tool
def validate_python(code: str) -> str:
    """Parse Python with ast. Returns 'ok' or the syntax error."""
    try:
        ast.parse(code); return "ok"
    except SyntaxError as e:
        return f"SyntaxError line {e.lineno}: {e.msg} | near: {e.text!r}"

@tool
def execute_glayout(code: str, run_name: str) -> str:
    """Execute glayout code in a sandbox dir. Returns the .gds path or an error string."""
    work = OUT_DIR / run_name; work.mkdir(parents=True, exist_ok=True)
    cwd = os.getcwd(); os.chdir(work)
    try:
        exec(compile(code, f"<{run_name}>", "exec"), {"__name__": "__main__"})
    except Exception:
        os.chdir(cwd); return "ExecError: " + traceback.format_exc().splitlines()[-1]
    os.chdir(cwd)
    g = sorted(work.glob("*.gds"), key=lambda p: p.stat().st_mtime, reverse=True)
    return str(g[0]) if g else "no .gds written"

@tool
def render_gds_to_png(gds_path: str) -> str:
    """Render a GDS to PNG via klayout. Returns the PNG path."""
    import klayout.lay as klay
    p = Path(gds_path); png = p.with_suffix(".png")
    lv = klay.LayoutView(); lv.load_layout(str(p), True); lv.max_hier(); lv.zoom_fit()
    lv.save_image(str(png), 1200, 800); return str(png)

print("tools: generate_glayout_code (← fine-tuned), continue_glayout_code (← fine-tuned),")
print("       validate_python, execute_glayout, render_gds_to_png")

tools: generate_glayout_code (← fine-tuned), continue_glayout_code (← fine-tuned),
       validate_python, execute_glayout, render_gds_to_png


## 4 · Subagents

Only one subagent is needed — a **planner** (pure reasoning, no tools). The coding is done by the `generate_glayout_code` *tool* (your fine-tuned model), which the main agent calls directly. This keeps the fine-tuned specialist focused on what it's good at.

In [10]:
from deepagents import SubAgent

# Planner: pure reasoning on the orchestrator model. No tools.
planner = SubAgent(
    name        = "planner",
    description = "Produces a short bullet plan of glayout primitives, ports, and routing for a requested circuit. No code.",
    system_prompt = (
        "You are an analog IC layout planner (glayout, gf180 PDK).\n"
        "Given a circuit request, output a short bullet plan:\n"
        "  1. Primitives (nmos/pmos/multiplier/two_nfet_interdigitized/diff_pair/current_mirror)\n"
        "  2. Key ports & shorts\n"
        "  3. Routing (c_route/L_route/straight_route)\n"
        "  4. Tap/guard rings if needed\n"
        "Be brief (<150 words). Do NOT write code — the coder tool handles that."
    ),
)

print("subagent: planner (orchestrator model, no tools)")

subagent: planner (orchestrator model, no tools)


## 5 · Hybrid agent: Ollama orchestrator + fine-tuned coder tool

The main agent runs on `qwen3.5:9b` (tool-calling). Its tools include `generate_glayout_code` and `continue_glayout_code` — both call **your fine-tuned 4B model** in-process. Memory + filesystem carry state between steps.

In [11]:
from deepagents import create_deep_agent

MAIN_INSTRUCTIONS = (
    "You orchestrate analog IC layout generation on gf180 PDK.\n"
    "The CODE is written by the fine-tuned `generate_glayout_code` tool — always use it, "
    "never write glayout code yourself.\n"
    "\n"
    "HARD RULES (do not violate):\n"
    "  - Build EXACTLY the circuit the user named. Never substitute, rename, or "
    "'improve' it into a different circuit. If the user says 'current mirror', you "
    "build a current mirror — not a ring oscillator, not an inverter, nothing else.\n"
    "  - When you call a tool or subagent, pass the USER'S REQUEST COPIED WORD-FOR-WORD. "
    "Do not paraphrase it and do not invent a new circuit description.\n"
    "  - Use the exact run_name the user gives, if any.\n"
    "\n"
    "Workflow for each user request (let R = the user's request, verbatim):\n"
    "  1. task('planner', R)                            → get a plan for R\n"
    "  2. generate_glayout_code(plan='<plan>')          → fine-tuned model writes code\n"
    "  3. validate_python(code='<code>')                → 'ok' or syntax error\n"
    "       - if it ends mid-line (truncated at 4096 tok): call\n"
    "         continue_glayout_code(partial_code='<last 30 lines>') and append the result,\n"
    "         then validate again. Repeat up to 4 times.\n"
    "  4. execute_glayout(code='<full code>', run_name='<unique>')   → .gds path\n"
    "  5. render_gds_to_png(gds_path='<that>')          → .png path\n"
    "  6. Report the .png path + one-line summary. The summary MUST name the same "
    "circuit the user asked for.\n"
    "\n"
    "Track used run_names in memory; never reuse one."
)

# NOTE: create_deep_agent already installs FilesystemMiddleware internally (and
# MemoryMiddleware when `memory=` is set). Passing FilesystemMiddleware() again
# via `middleware=` puts a *second* instance into the same stack, which
# create_agent rejects with "Please remove duplicate middleware instances."
# So no `middleware` arg here — filesystem + memory are wired up automatically.
agent = create_deep_agent(
    model         = llm,                       # orchestrator = qwen3.5:9b
    tools         = [generate_glayout_code,    # ← your fine-tuned model
                     continue_glayout_code,    # ← your fine-tuned model
                     validate_python, execute_glayout, render_gds_to_png],
    system_prompt = MAIN_INSTRUCTIONS,
    subagents     = [planner],
    memory        = ["plan", "runs_used", "last_code"],
)

print("✓ Hybrid agent compiled: qwen3.5:9b orchestrates, fine-tuned 4B writes the code.")

✓ Hybrid agent compiled: qwen3.5:9b orchestrates, fine-tuned 4B writes the code.


## 6 · Run with streaming

Stream every event so you watch each subagent's decision in real time.

In [12]:
from IPython.display import Image as IPyImage, display

def run_agent(user_prompt: str):
    print(f"USER: {user_prompt}\n")
    state = {"messages": [{"role": "user", "content": user_prompt}]}
    final = None
    for event in agent.stream(state, stream_mode="values"):
        msgs = event.get("messages", [])
        if msgs:
            last = msgs[-1]
            role    = getattr(last, "type", getattr(last, "role", "?"))
            content = getattr(last, "content", "")
            if isinstance(content, list):
                content = " | ".join(str(c)[:120] for c in content)
            print(f"  [{role:10s}] {str(content)[:300]}")
        final = event
    return final

final = run_agent("Generate a current mirror on gf180 PDK. Use run_name='01_current_mirror'.")

USER: Generate a current mirror on gf180 PDK. Use run_name='01_current_mirror'.

  [human     ] Generate a current mirror on gf180 PDK. Use run_name='01_current_mirror'.
  [human     ] Generate a current mirror on gf180 PDK. Use run_name='01_current_mirror'.
  [ai        ] 
  [tool      ] **Current Mirror Circuit Plan (gf180 PDK)**

**1. Primitives:**
- Two NMOS transistors (M1, M2) with identical W/L ratios for current mirroring
- Optional PMOS current source for biasing reference current
- Gate-drain short on reference transistor (diode-connected)

**2. Key Ports & Shorts:**
- **In
  [ai        ] 


KeyboardInterrupt: 

## 7 · Display the result

Walk the agent_outputs/ dir for the latest PNG and show it.

In [ ]:
pngs = sorted(OUT_DIR.rglob("*.png"), key=lambda p: p.stat().st_mtime, reverse=True)
if pngs:
    print(f"Latest layout: {pngs[0]}")
    display(IPyImage(filename=str(pngs[0])))
else:
    print("No PNG produced yet — check the agent stream above for errors.")

## 8 · Memory carries across runs

The same agent instance — `runs_used` memory now contains the previous run. Generate a second circuit and the agent will pick a different `run_name` automatically.

In [ ]:
final2 = run_agent("Now generate a differential pair on gf180 PDK.")
pngs = sorted(OUT_DIR.rglob("*.png"), key=lambda p: p.stat().st_mtime, reverse=True)
if pngs:
    display(IPyImage(filename=str(pngs[0])))